In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# =========================================
# Load Data
# =========================================
df = pd.read_excel("business_regression_data.xlsx")

# =========================================
# Create Dummy Variables
# =========================================
df_model = pd.get_dummies(
    df,
    columns=["region", "store_type"],
    drop_first=True,
    dtype=int
)

# =========================================
# Select Variables
# =========================================
predictors = [
    'marketing_spend',
    'footfall',
    'inventory_availability_pct',
    'customer_rating'
]

dummy_cols = [
    col for col in df_model.columns
    if col.startswith('region_') or col.startswith('store_type_')
]

predictors += dummy_cols

X = df_model[predictors]
y = df_model['monthly_sales']

# Remove NaN rows
combined = pd.concat([X, y], axis=1).dropna()

X = combined.drop(columns=['monthly_sales'])
y = combined['monthly_sales']

X = sm.add_constant(X)

# =========================================
# Fit Model
# =========================================
model = sm.OLS(y, X).fit()

# =========================================
# Predictions and Residuals
# =========================================
predicted_sales = model.predict(X)

residual_df = pd.DataFrame({
    "Actual_Sales": y,
    "Predicted_Sales": predicted_sales,
    "Residual": y - predicted_sales
})

# =========================================
# Largest Positive Residuals
# =========================================
top_positive = residual_df.nlargest(5, "Residual")

# =========================================
# Largest Negative Residuals
# =========================================
top_negative = residual_df.nsmallest(5, "Residual")

print("\nTop 5 Positive Residuals")
print(top_positive)

print("\nTop 5 Negative Residuals")
print(top_negative)

# =========================================
# Save to Workbook
# =========================================
with pd.ExcelWriter(
    "regression_workbook.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    residual_df.to_excel(
        writer,
        sheet_name="Predictions_Residuals",
        index=False
    )

print("\nWorkbook updated.")

# =========================================
# Create residual_analysis.md
# =========================================
with open("residual_analysis.md", "w", encoding="utf-8") as f:

    f.write("# Residual Analysis\n\n")

    f.write("## Largest Positive Residuals\n\n")
    f.write(top_positive.to_markdown(index=False))

    f.write("\n\n")

    f.write("## Largest Negative Residuals\n\n")
    f.write(top_negative.to_markdown(index=False))

    f.write("\n\n")

    f.write("## Business Interpretation\n\n")

    f.write("- Positive residuals indicate stores where actual sales exceeded model predictions.\n")
    f.write("- Negative residuals indicate stores where actual sales were lower than predicted.\n")
    f.write("- Large residuals suggest that additional factors not included in the model may affect sales.\n")
    f.write("- The model may under-predict some stores and over-predict others.\n")
    f.write("- Regression captures association and cannot explain every variation in sales.\n")

print("\nresidual_analysis.md created successfully.")



Top 5 Positive Residuals
     Actual_Sales  Predicted_Sales       Residual
111     713611.16    595606.532257  118004.627743
290     813316.71    703979.884959  109336.825041
74      788087.68    697782.069844   90305.610156
103     625514.04    535442.874370   90071.165630
199     735786.64    651842.471638   83944.168362

Top 5 Negative Residuals
    Actual_Sales  Predicted_Sales       Residual
66     685379.08    844031.931519 -158652.851519
89     627171.90    769659.417291 -142487.517291
44     595467.60    716364.860950 -120897.260950
25     800451.94    915237.108410 -114785.168410
32     641865.03    747558.013688 -105692.983688

Workbook updated.

residual_analysis.md created successfully.
